# 02 · Choose the corpus scope (no BigQuery, no quota used)

Uses only the two files notebook 01 saved to Google Drive. For several candidate scopes it counts:

- **Corpus size:** patent families (one granted patent per family)
- **Test cases:** recent G06N documents (filed 2022 or later) whose **examiner** citations (category `SEA`) point to patents inside that corpus

The best scope has about 20,000 families **and** many test cases with several examiner citations each.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd

DRIVE_DIR = "/content/drive/MyDrive/patent-search"
meta = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_meta.parquet")
cit = pd.read_parquet(f"{DRIVE_DIR}/g06n_us_citations.parquet")

meta["year"] = meta["filing_date"] // 10000
meta["is_grant"] = meta["kind_code"].str.startswith("B")
meta["first_cpc"] = meta["first_cpc"].fillna("")
family_of = dict(zip(meta["publication_number"], meta["family_id"]))

# Examiner citations only, mapped to the cited patent's family
cit = cit[cit["category"].fillna("").str.contains("SEA")].copy()
cit["cited_family"] = cit["cited"].map(family_of)
cit["citing_family"] = cit["publication_number"].map(family_of)
cit = cit.dropna(subset=["cited_family", "citing_family"])
print(f"{len(meta):,} documents, {len(cit):,} examiner citations between G06N documents")

In [ ]:
def corpus_families(scope):
    """Families in a scope: granted patents filed 2012-2021 matching the scope rule."""
    m = meta[meta.is_grant & meta.year.between(2012, 2021)]
    return set(m.loc[scope(m), "family_id"])

def test_cases(families):
    """Families filed 2022+ (outside the corpus) and their examiner citations into it."""
    recent = set(meta.loc[meta.year >= 2022, "family_id"]) - families
    c = cit[cit.citing_family.isin(recent) & cit.cited_family.isin(families)]
    return c.groupby("citing_family")["cited_family"].nunique()

SCOPES = {
    "G06N is the main code":            lambda m: m.first_cpc.str.startswith("G06N"),
    "Main code G06N3 (neural networks)":  lambda m: m.first_cpc.str.startswith("G06N3/"),
    "Main code G06N20 (machine learning)": lambda m: m.first_cpc.str.startswith("G06N20/"),
    "Main code G06N3 or G06N20":        lambda m: m.first_cpc.str.match(r"G06N(3|20)/"),
    "Any G06N code, filed 2018-2021":   lambda m: m.year >= 2018,
}

rows = []
for name, rule in SCOPES.items():
    fams = corpus_families(rule)
    tc = test_cases(fams)
    rows.append({"scope": name, "families": len(fams),
                 "test cases (>=1 cite)": len(tc), "(>=2 cites)": int((tc >= 2).sum()),
                 "(>=3 cites)": int((tc >= 3).sum()), "avg cites": round(tc.mean(), 1) if len(tc) else 0})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Size of each scope by filing year, to see where a 20,000 cut would fall
for name in ["G06N is the main code", "Main code G06N3 or G06N20"]:
    rule = SCOPES[name]
    m = meta[meta.is_grant & meta.year.between(2012, 2021)]
    m = m[rule(m)].drop_duplicates("family_id")
    print(f"\n{name}: families by filing year")
    print(m.groupby("year")["family_id"].nunique().to_string())